# 5.1 — Phasor GAN: Generative Adversarial Network for Timeseries

This notebook demonstrates a **Generative Adversarial Network (GAN)** where both the Generator and Discriminator are phasor circuits. The Generator maps random latent noise through trainable phase shifts and DFT mixing to produce synthetic timeseries samples, while the Discriminator classifies samples as real or fake using a separate phasor circuit with shift + mix + DFT layers.

In [ ]:
import sys
import os
sys.path.append(os.path.abspath('..'))
import PhasorFlow as pf
from PhasorFlow.models.gan import PhasorGAN
import numpy as np
import matplotlib.pyplot as plt
import torch

np.random.seed(42)
torch.manual_seed(42)

## 1. Synthetic Timeseries Data

We generate a composite waveform signal: $\sin(t) + 0.5\cos(3t) + 0.3\sin(5t)$, then window it into fixed-length samples.

In [ ]:
SEQ_LENGTH = 8
NUM_SAMPLES = 100

# Generate composite waveform
t_total = np.linspace(0, 8 * np.pi, NUM_SAMPLES + SEQ_LENGTH)
signal = (
    np.sin(t_total)
    + 0.5 * np.cos(3 * t_total)
    + 0.3 * np.sin(5 * t_total)
)
signal = signal / np.max(np.abs(signal))  # Normalize to [-1, 1]
signal += np.random.normal(0, 0.02, len(signal))
signal = np.clip(signal, -0.99, 0.99)

# Window into samples
X_real = np.array([signal[i:i+SEQ_LENGTH] for i in range(NUM_SAMPLES)], dtype=np.float32)
X_real_tensor = torch.tensor(X_real, dtype=torch.float32)

print(f"Real data shape: {X_real.shape}")
print(f"Value range: [{X_real.min():.3f}, {X_real.max():.3f}]")

# Plot a few real samples
plt.figure(figsize=(10, 4))
for i in range(8):
    plt.plot(X_real[i], alpha=0.6)
plt.title('Real Timeseries Samples')
plt.xlabel('Time Step')
plt.ylabel('Amplitude')
plt.tight_layout()
plt.show()

## 2. Phasor GAN Architecture

Our GAN uses two phasor circuits:

**Generator** ($G$):
$$z \in [-\pi, \pi]^T \;\xrightarrow{\text{Shift + DFT} \times L}\; \sin(\hat{\theta}) \;\to\; \text{fake sample}$$

**Discriminator** ($D$):
$$x \;\xrightarrow{\arcsin}\; \theta \;\xrightarrow{\text{Shift + Mix + DFT} \times L}\; P(\text{real})$$

Both use trainable phase shifts as variational parameters, with DFT for global token/thread mixing.

In [ ]:
# Initialize the Phasor GAN
gan = PhasorGAN(
    seq_length=SEQ_LENGTH,
    num_layers_g=2,
    num_layers_d=2,
)
print(gan)

## 3. Adversarial Training

Standard minimax GAN training:
- **D step**: maximize $\log D(x_{\text{real}}) + \log(1 - D(G(z)))$
- **G step**: maximize $\log D(G(z))$

In [ ]:
history = gan.fit(
    X_real_tensor,
    epochs=60,
    batch_size=10,
    lr_g=0.01,
    lr_d=0.01,
    print_every=10,
)

## 4. Training Curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Loss curves
axes[0].plot(history['d_loss'], label='D Loss', color='tab:blue')
axes[0].plot(history['g_loss'], label='G Loss', color='tab:red')
axes[0].set_title('GAN Training Loss')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].legend()

# Discriminator confidence
axes[1].plot(history['d_real'], label='D(real)', color='tab:green')
axes[1].plot(history['d_fake'], label='D(fake)', color='tab:orange')
axes[1].axhline(y=0.5, color='gray', linestyle=':', alpha=0.5)
axes[1].set_title('Discriminator Confidence')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Probability')
axes[1].legend()

plt.tight_layout()
plt.show()

## 5. Generated vs Real Samples

We use the trained Generator to produce synthetic timeseries and compare them visually against real data.

In [ ]:
# Generate fake samples
fake_samples = gan.generate(num_samples=8)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Real samples
for i in range(8):
    axes[0].plot(X_real[i], alpha=0.6, color='tab:blue')
axes[0].set_title('Real Timeseries')
axes[0].set_xlabel('Time Step')
axes[0].set_ylabel('Amplitude')
axes[0].set_ylim(-1.5, 1.5)

# Generated samples
for i in range(8):
    axes[1].plot(fake_samples[i].numpy(), alpha=0.6, color='tab:red')
axes[1].set_title('Generated Timeseries (Phasor GAN)')
axes[1].set_xlabel('Time Step')
axes[1].set_ylabel('Amplitude')
axes[1].set_ylim(-1.5, 1.5)

plt.tight_layout()
plt.show()

## 6. Summary Statistics

In [ ]:
print("=" * 50)
print("  Phasor GAN — Summary")
print("=" * 50)
print(f"Sequence length:        {SEQ_LENGTH}")
print(f"Generator layers:       {gan.num_layers_g}")
print(f"Discriminator layers:   {gan.num_layers_d}")
print(f"Generator params:       {gan.generator.weights.numel()}")
print(f"Discriminator params:   {gan.discriminator.weights.numel()}")
print(f"Final D(real):          {history['d_real'][-1]:.4f}")
print(f"Final D(fake):          {history['d_fake'][-1]:.4f}")
print(f"Final D Loss:           {history['d_loss'][-1]:.4f}")
print(f"Final G Loss:           {history['g_loss'][-1]:.4f}")

# Distribution comparison
real_means = X_real.mean(axis=1)
fake_means = fake_samples.numpy().mean(axis=1)
print(f"\nReal sample mean:       {real_means.mean():.4f} ± {real_means.std():.4f}")
print(f"Fake sample mean:       {fake_means.mean():.4f} ± {fake_means.std():.4f}")